# Phase 5 — Impact Analysis + Evidence Consolidation

## 1. Purpose

This notebook turns the validated December 2025 Lower Fraser flood-change result into a defensible impact case study. It asks where mapped flood expansion occurred, which mapped assets lie in or near it, and whether the geography is broadly consistent with official event reporting.

The analysis is deliberately conservative: spatial overlap means **potential exposure**, not damage, closure, flood depth, population impact, or causation. Phase 6 is not implemented here.

## 2. Inputs inherited from Phase 4

Phase 5 reuses the fixed Phase 4 AOI/grid, semantic class interpretation, validated metrics, Folly Lake control context, recovery-period Sentinel-2 comparison, and temporary public-data cache. It does **not** repeat candidate-event screening or the complete seven-date workflow.

Only the deterministic 12 December near-event and 21 December recovery EGS products are loaded to reconstruct the masks needed for impact geometry. If the Phase 4 temporary cache is absent, only those two public ZIPs are downloaded.

## 3. Final event / AOI recap

In [ ]:
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import io, json, shutil, tempfile, zipfile

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
from pyproj import Transformer
from rasterio.features import geometry_mask, rasterize, shapes
from rasterio.transform import array_bounds, from_origin
from shapely.geometry import LineString, box, mapping, shape
from shapely.ops import transform as shapely_transform, unary_union
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 13, "axes.labelsize": 10})

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
OUTPUT_DIR = REPO / "data" / "processed" / "phase5"
ASSET_DIR = REPO / "assets" / "phase5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ASSET_DIR.mkdir(parents=True, exist_ok=True)

WORKING_CRS = "EPSG:32610"
AOI_BOUNDS_UTM = (563960, 5443240, 569960, 5449240)
PIXEL_SIZE = 20
PIXEL_AREA_HA = 0.04
GRID_METRES = 1000
CONTROL_CONTEXT_PP = 5.88
AOI_UTM = box(*AOI_BOUNDS_UTM)
to_wgs84 = Transformer.from_crs(WORKING_CRS, "EPSG:4326", always_xy=True).transform
AOI_WGS84 = shapely_transform(to_wgs84, AOI_UTM)
AOI_BBOX = AOI_WGS84.bounds
TARGET_TRANSFORM = from_origin(AOI_BOUNDS_UTM[0], AOI_BOUNDS_UTM[3], PIXEL_SIZE, PIXEL_SIZE)
TARGET_SHAPE = (300, 300)
AOI_MASK = geometry_mask([mapping(AOI_UTM)], out_shape=TARGET_SHAPE, transform=TARGET_TRANSFORM, invert=True)

phase4_metrics = pd.DataFrame([
    ("AOI area", 36.00, "km²"),
    ("Semantic permanent-water baseline", 971.88, "ha"),
    ("Near-event total water", 1026.80, "ha"),
    ("Mapped event water gain", 54.92, "ha"),
    ("Recovery total water", 993.52, "ha"),
    ("Near-event to recovery mapped water loss", 35.44, "ha"),
    ("Maximum local grid increase", 21.50, "percentage points"),
    ("Grid cells above 5.88 pp control context", 2, "cells"),
    ("RCM/Sentinel-2 recovery F1", 0.790, "score"),
    ("RCM/Sentinel-2 recovery IoU", 0.653, "score"),
], columns=["metric", "value", "unit"])
display(phase4_metrics)
print("Fixed CRS:", WORKING_CRS, "| bounds:", AOI_BOUNDS_UTM, "| area:", AOI_UTM.area / 1e6, "km²")

## 4. High-priority flood-change zones

In [ ]:
EGS_BASE = "https://data.eodms-sgdot.nrcan-rncan.gc.ca/public/EGS/2025/Flood/CAN/BC/"
EGS_FILES = {
    "near-event": "Flood_CAN_BC_LowerFraser_20251212_140824.zip",
    "recovery": "Flood_CAN_BC_LowerFraser_20251221_141646.zip",
}
EGS_CACHE = Path(tempfile.gettempdir()) / "earth-in-motion-phase4-egs"
EGS_CACHE.mkdir(parents=True, exist_ok=True)

def ensure_egs_product(filename):
    folder = EGS_CACHE / Path(filename).stem
    if list(folder.rglob("Flood_*.shp")) and list(folder.rglob("Footprint_*.shp")):
        return folder
    request = Request(EGS_BASE + filename, headers={"User-Agent": "earth-in-motion-hackathon/1.0"})
    with urlopen(request, timeout=180) as response:
        payload = response.read()
    folder.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        archive.extractall(folder)
    return folder

def rasterize_geometry(geometry):
    if geometry is None or geometry.is_empty:
        return np.zeros(TARGET_SHAPE, dtype=np.uint8)
    return rasterize([(mapping(geometry), 1)], out_shape=TARGET_SHAPE, transform=TARGET_TRANSFORM,
                     fill=0, dtype=np.uint8, all_touched=False)

scenes = {}
for label, filename in EGS_FILES.items():
    folder = ensure_egs_product(filename)
    flood = gpd.read_file(next(folder.rglob("Flood_*.shp"))).clip(AOI_WGS84).to_crs(WORKING_CRS)
    footprint = gpd.read_file(next(folder.rglob("Footprint_*.shp"))).clip(AOI_WGS84).to_crs(WORKING_CRS)
    valid = rasterize_geometry(footprint.geometry.union_all()).astype(bool) & AOI_MASK
    permanent = rasterize_geometry(flood.loc[flood["class"].eq(1)].geometry.union_all()).astype(bool) & valid
    flood_mask = rasterize_geometry(flood.loc[flood["class"].eq(2)].geometry.union_all()).astype(bool) & valid
    scenes[label] = {"valid": valid, "permanent": permanent, "flood": flood_mask}

valid = scenes["near-event"]["valid"] & scenes["recovery"]["valid"]
baseline = scenes["near-event"]["permanent"] & valid
event = baseline | (scenes["near-event"]["flood"] & valid)
recovery = baseline | (scenes["recovery"]["flood"] & valid)
gain = valid & ~baseline & event
loss = valid & event & ~recovery

gain_parts = [shape(geom) for geom, value in shapes(gain.astype(np.uint8), mask=gain, transform=TARGET_TRANSFORM) if value == 1]
loss_parts = [shape(geom) for geom, value in shapes(loss.astype(np.uint8), mask=loss, transform=TARGET_TRANSFORM) if value == 1]
gain_union = unary_union(gain_parts)
loss_union = unary_union(loss_parts)

reconstructed = {
    "baseline_ha": baseline.sum() * PIXEL_AREA_HA,
    "event_ha": event.sum() * PIXEL_AREA_HA,
    "gain_ha": gain.sum() * PIXEL_AREA_HA,
    "recovery_ha": recovery.sum() * PIXEL_AREA_HA,
    "loss_ha": loss.sum() * PIXEL_AREA_HA,
}
expected = {"baseline_ha": 971.88, "event_ha": 1026.80, "gain_ha": 54.92, "recovery_ha": 993.52, "loss_ha": 35.44}
for key, value in expected.items():
    assert abs(reconstructed[key] - value) < 0.01, (key, reconstructed[key], value)
print("Minimal reconstruction matches Phase 4:", reconstructed)

## 5. Road exposure analysis

In [ ]:
def get_overpass_payload():
    phase5_cache = Path(tempfile.gettempdir()) / "earth-in-motion-phase5-roads.json"
    phase4_cache = Path(tempfile.gettempdir()) / "phase4_roads.json"
    if phase5_cache.exists():
        return json.loads(phase5_cache.read_text())
    if phase4_cache.exists():
        payload = json.loads(phase4_cache.read_text())
        phase5_cache.write_text(json.dumps(payload))
        return payload
    west, south, east, north = AOI_BBOX
    query = f'[out:json][timeout:90];way["highway"]({south},{west},{north},{east});out tags geom;'
    endpoints = ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter"]
    for endpoint in endpoints:
        try:
            request = Request(endpoint, data=urlencode({"data": query}).encode(), headers={"User-Agent": "earth-in-motion-hackathon/1.0"})
            with urlopen(request, timeout=120) as response:
                payload = json.load(response)
            phase5_cache.write_text(json.dumps(payload))
            return payload
        except Exception as exc:
            print("Overpass endpoint failed:", endpoint, type(exc).__name__)
    raise RuntimeError("No Overpass endpoint returned road data")

road_rows = []
for element in get_overpass_payload().get("elements", []):
    coords = [(node["lon"], node["lat"]) for node in element.get("geometry", [])]
    if len(coords) < 2:
        continue
    tags = element.get("tags", {})
    road_rows.append({
        "osm_id": element["id"], "name": tags.get("name"), "ref": tags.get("ref"),
        "highway": tags.get("highway"), "bridge": tags.get("bridge"),
        "surface": tags.get("surface"), "access": tags.get("access"), "geometry": LineString(coords),
    })

roads = gpd.GeoDataFrame(road_rows, geometry="geometry", crs=4326).to_crs(WORKING_CRS)
roads["geometry"] = roads.geometry.intersection(AOI_UTM)
roads = roads.loc[~roads.geometry.is_empty].copy()
roads["length_in_aoi_m"] = roads.length
roads["flood_intersection_m"] = roads.geometry.intersection(gain_union).length
roads["distance_to_gain_m"] = roads.geometry.distance(gain_union)
roads["direct_intersection"] = roads.flood_intersection_m.gt(0)
for distance in (50, 100, 250):
    roads[f"within_{distance}m"] = roads.distance_to_gain_m.le(distance)
major_groups = ["motorway", "trunk", "primary", "secondary", "tertiary", "residential", "service", "unclassified"]
roads["road_group"] = roads.highway.where(roads.highway.isin(major_groups), "other")
direct_roads = roads.loc[roads.direct_intersection].sort_values("flood_intersection_m", ascending=False).copy()
road_class_summary = direct_roads.groupby("road_group", dropna=False).agg(
    ways=("osm_id", "count"), intersection_m=("flood_intersection_m", "sum")
).sort_values("intersection_m", ascending=False)
display(road_class_summary.round(2))
display(direct_roads[["osm_id", "name", "ref", "highway", "bridge", "surface", "access", "length_in_aoi_m", "flood_intersection_m"]].round(2))
print(f"Refined AOI clip: {len(roads)} ways; {len(direct_roads)} direct intersections; {direct_roads.flood_intersection_m.sum()/1000:.3f} km.")
print("The 232-way clipped count refines Phase 4's 238 bbox-return count; the direct result is unchanged.")

## 6. Bridge exposure analysis

In [ ]:
bridges = roads.loc[roads.bridge.notna() & ~roads.bridge.isin(["no", "false"])].copy()
bridge_columns = ["osm_id", "name", "ref", "highway", "bridge", "surface", "access", "length_in_aoi_m", "flood_intersection_m", "distance_to_gain_m"]
display(bridges[bridge_columns].sort_values("distance_to_gain_m").round(2))
print(f"{len(bridges)} bridge-tagged ways in the clipped AOI; {int(bridges.direct_intersection.sum())} directly intersects mapped gain.")
print("The direct feature is an OSM way named Lougheed Highway, ref BC 7; this is potential exposure, not structural damage.")

## 7. Community / populated-place context

In [ ]:
PLACE_URL = "https://ised-isde.canada.ca/app/scr/sittibc/web/api/openData/MAG_EXO.CSV"
place_cache = Path(tempfile.gettempdir()) / "earth-in-motion-phase5-places.csv"
if not place_cache.exists():
    request = Request(PLACE_URL, headers={"User-Agent": "earth-in-motion-hackathon/1.0"})
    with urlopen(request, timeout=120) as response:
        place_cache.write_bytes(response.read())
places_raw = pd.read_csv(place_cache, encoding="latin1")
places_raw = places_raw.loc[
    places_raw.Province.eq("BC") &
    places_raw.Longitude.between(AOI_BBOX[0] - 0.30, AOI_BBOX[2] + 0.30) &
    places_raw.Latitude.between(AOI_BBOX[1] - 0.20, AOI_BBOX[3] + 0.20)
].copy()
places = gpd.GeoDataFrame(
    places_raw.rename(columns={"Name_en": "place_name"}),
    geometry=gpd.points_from_xy(places_raw.Longitude, places_raw.Latitude), crs=4326,
).to_crs(WORKING_CRS)
places["inside_aoi"] = places.geometry.within(AOI_UTM)
places["distance_to_gain_m"] = places.geometry.distance(gain_union)
for distance in (50, 100, 250):
    places[f"within_{distance}m"] = places.distance_to_gain_m.le(distance)
places = places.sort_values("distance_to_gain_m").drop_duplicates("place_name").reset_index(drop=True)
display(places[["PNuid_NLidu", "place_name", "Latitude", "Longitude", "inside_aoi", "distance_to_gain_m"]].head(15).round(2))
print("Nearest official place point:", places.iloc[0].place_name, f"({places.iloc[0].distance_to_gain_m:.0f} m from mapped gain).")
print("Point locations are context only; no population-exposure counts are inferred.")

## 8. Agricultural / land-use exposure

In [ ]:
ALR_ENDPOINT = "https://delivery.maps.gov.bc.ca/arcgis/rest/services/whse/bcgw_pub_whse_legal_admin_boundaries/MapServer/23/query"
alr_params = {
    "where": "1=1", "geometry": ",".join(map(str, AOI_BBOX)), "geometryType": "esriGeometryEnvelope",
    "inSR": "4326", "outSR": "4326", "outFields": "ALR_POLY_ID,STATUS,FEATURE_CODE",
    "returnGeometry": "true", "f": "geojson",
}
request = Request(ALR_ENDPOINT + "?" + urlencode(alr_params), headers={"User-Agent": "earth-in-motion-hackathon/1.0"})
with urlopen(request, timeout=120) as response:
    alr_payload = json.load(response)
alr = gpd.GeoDataFrame.from_features(alr_payload["features"], crs=4326).to_crs(WORKING_CRS).clip(AOI_UTM)
alr_union = alr.geometry.union_all()
alr_gain = gain_union.intersection(alr_union)
alr_gain_ha = alr_gain.area / 10_000
alr_gain_pct = 100 * alr_gain.area / gain_union.area
print(f"ALR designated land in AOI: {alr_union.area/10_000:.2f} ha")
print(f"Mapped flood gain intersecting ALR designation: {alr_gain_ha:.2f} ha ({alr_gain_pct:.1f}% of mapped gain)")
print("ALR is a legal land designation, not a crop map; this does not prove cropped acreage or agricultural damage.")

## 9. Distance-to-flood analysis

In [ ]:
proximity = pd.DataFrame([
    ("Road ways", int(roads.direct_intersection.sum()), int(roads.within_50m.sum()), int(roads.within_100m.sum()), int(roads.within_250m.sum())),
    ("Bridge-tagged ways", int(bridges.direct_intersection.sum()), int(bridges.within_50m.sum()), int(bridges.within_100m.sum()), int(bridges.within_250m.sum())),
    ("Official place points", int(places.distance_to_gain_m.eq(0).sum()), int(places.within_50m.sum()), int(places.within_100m.sum()), int(places.within_250m.sum())),
], columns=["asset_type", "direct_intersection", "within_50m", "within_100m", "within_250m"])
display(proximity)
print("Buffer counts are cumulative and include direct intersections. They measure proximity only.")

## 10. Grid-level impact ranking

In [ ]:
left, bottom, right, top = array_bounds(*TARGET_SHAPE, TARGET_TRANSFORM)
grid_rows, grid_id = [], 0
for x0 in np.arange(np.floor(left / GRID_METRES) * GRID_METRES, right, GRID_METRES):
    for y0 in np.arange(np.floor(bottom / GRID_METRES) * GRID_METRES, top, GRID_METRES):
        cell = box(x0, y0, x0 + GRID_METRES, y0 + GRID_METRES)
        cell_mask = geometry_mask([mapping(cell)], out_shape=TARGET_SHAPE, transform=TARGET_TRANSFORM, invert=True) & AOI_MASK
        cell_valid = cell_mask & valid
        if not cell_valid.any():
            continue
        grid_id += 1
        pre_ratio = baseline[cell_valid].mean() * 100
        event_ratio = event[cell_valid].mean() * 100
        recovery_ratio = recovery[cell_valid].mean() * 100
        gain_area = (gain & cell_mask).sum() * PIXEL_AREA_HA
        cell_gain = gain_union.intersection(cell)
        if cell_gain.is_empty:
            road_hits, bridge_hits = roads.iloc[0:0], bridges.iloc[0:0]
            nearest_name, nearest_distance = None, np.nan
        else:
            road_hits = roads.loc[roads.geometry.intersects(cell_gain)]
            bridge_hits = bridges.loc[bridges.geometry.intersects(cell_gain)]
            nearest_idx = places.geometry.distance(cell_gain).idxmin()
            nearest_name = places.loc[nearest_idx, "place_name"]
            nearest_distance = places.loc[nearest_idx].geometry.distance(cell_gain)
        delta = event_ratio - pre_ratio
        category = "high observed expansion" if delta > CONTROL_CONTEXT_PP else ("moderate observed expansion" if delta >= 2 else ("low observed expansion" if gain_area > 0 else "no mapped expansion"))
        temporal = "recovery observed" if gain_area > 0 and recovery_ratio < event_ratio else ("persistent expansion" if gain_area > 0 else "no mapped event gain")
        note_bits = []
        if delta > CONTROL_CONTEXT_PP: note_bits.append("above control-site context")
        if len(road_hits): note_bits.append("road intersection")
        if len(bridge_hits): note_bits.append("bridge-tag intersection")
        grid_rows.append({
            "grid_id": f"G{grid_id:03d}", "baseline_water_ratio_pct": pre_ratio,
            "event_water_ratio_pct": event_ratio, "recovery_water_ratio_pct": recovery_ratio,
            "event_minus_baseline_pp": delta, "event_gain_area_ha": gain_area,
            "interpretation": category, "temporal_class": temporal,
            "road_count_direct": len(road_hits),
            "road_intersection_m": road_hits.geometry.intersection(cell_gain).length.sum() if len(road_hits) else 0,
            "bridge_count_direct": len(bridge_hits), "nearest_community": nearest_name,
            "community_distance_to_cell_gain_m": nearest_distance,
            "alr_overlap_ha": cell_gain.intersection(alr_union).area / 10_000,
            "evidence_notes": "; ".join(note_bits) or "mapped change only",
            "geometry": cell,
        })
grid = gpd.GeoDataFrame(grid_rows, geometry="geometry", crs=WORKING_CRS)
priority = grid.loc[
    grid.event_gain_area_ha.gt(0) &
    (grid.event_minus_baseline_pp.gt(CONTROL_CONTEXT_PP) | grid.road_count_direct.gt(0) | grid.bridge_count_direct.gt(0))
].sort_values(["event_minus_baseline_pp", "event_gain_area_ha"], ascending=False).copy()
display(priority.drop(columns="geometry").round(3))
assert set(["G042", "G015"]).issubset(set(priority.grid_id))
print("No composite risk score is used; ordering is descriptive by observed percentage-point expansion and area.")

## 11. Official event-impact evidence

In [ ]:
official_impacts = pd.DataFrame([
    ("2025-12-10", "EmergencyInfoBC", "Flood warning for the Sumas River and Lower Fraser tributaries, including Chilliwack, Abbotsford and Hope.", "https://www.emergencyinfobc.gov.bc.ca/event/floodwatch-fraservalley-09122025/"),
    ("2025-12-11", "City of Abbotsford", "Highway 1 closed between Sumas Way and No. 3 Road; evacuation orders covered 82 Huntingdon and 371 Sumas Prairie West properties, with 1,069 properties on alert.", "https://www.abbotsford.ca/city-hall/news-media/highway-1-closed-between-sumas-way-and-no-3-road-and-evacuation-orders"),
    ("2025-12-13", "Government of British Columbia", "Provincial update documented Highway 1 reopening work and remaining water on eastbound lanes in the wider affected region.", "https://news.gov.bc.ca/releases/2025TT0126-001250"),
    ("2025-12-23", "City of Abbotsford", "Event summary reported 485 properties under evacuation order, 1,069 on alert, property damage and livestock loss.", "https://www.abbotsford.ca/council/your-council-community/blog/abbotsford-shows-resilience-and-compassion-face-disaster"),
    ("2025-12-10", "Fraser Valley Regional District", "Wilson Road evacuation order described immediate danger from flooding due to the Chilliwack River.", "https://www.emergencyinfobc.gov.bc.ca/app/uploads/sites/893/2025/12/2025-12-10-Evacuation-ORDER-and-ALERT-Wilson-Road-Area.pdf"),
], columns=["date", "source", "documented_context", "url"])
display(official_impacts)
print("These reports establish regional event context; they are not pixel labels and are not assigned to a specific mapped road without exact official geometry.")

## 12. Spatial consistency with documented impacts

In [ ]:
consistency = pd.DataFrame([
    ("Regional geography", "The AOI lies in the Lower Fraser / Fraser Valley region named by official warnings.", "Broad regional consistency, not exact impact confirmation."),
    ("Hydrologic pattern", "Mapped gain follows existing water corridors and low-lying ALR-designated land.", "Spatially plausible open-water expansion; no depth estimate."),
    ("Infrastructure", "Fourteen clipped OSM road ways intersect mapped gain; BC 7 is among them.", "Potential exposure only; the compact AOI does not include the reported Highway 1 closure segment."),
    ("Temporal behavior", "Water gain declines from the 12 Dec event observation toward 21 Dec recovery.", "Broadly coherent with a short-duration flood response."),
], columns=["test", "observation", "interpretation"])
display(consistency)

## 13. Evidence-chain construction

In [ ]:
evidence_chain = pd.DataFrame([
    ("Official warning", "EmergencyInfoBC", "2025-12-10", "Fraser Valley / Lower Fraser flood warning", "Establishes event and region", "Not a pixel label"),
    ("Near-event flood map", "NRCan EGS, RCM-derived", "2025-12-12", "Permanent-water and open-water flood classes", "Maps event-period water", "Moderate confidence; open water emphasis"),
    ("Change detection", "Phase 4/5 fixed-grid analysis", "2025-12-12", "54.92 ha outside semantic permanent-water baseline", "Quantifies mapped gain", "Depends on EGS classes and 20 m rasterization"),
    ("Temporal recession", "NRCan EGS sequence", "2025-12-12 to 21", "35.44 ha mapped loss by recovery observation", "Supports recovery interpretation", "Not a long-term trend"),
    ("Independent EO", "Sentinel-2 L2A", "2025-12-27", "RCM/optical recovery F1 0.790; IoU 0.653", "Independent recovery support", "6.2-day timing offset"),
    ("Infrastructure exposure", "OpenStreetMap / Overpass", "retrieved 2026-10-03", "14 road ways and 1 bridge-tagged way intersect gain", "Adds asset context", "Completeness/tagging vary; no damage proof"),
    ("Agricultural context", "BC Agricultural Land Reserve", "retrieved 2026-10-03", f"{alr_gain_ha:.2f} ha of mapped gain intersects ALR designation", "Adds authoritative land designation", "Not actual crop or damage mapping"),
    ("Regional impact", "Abbotsford / B.C. official reports", "2025-12-11 to 23", "Evacuations, road interruption, property and livestock impacts", "Corroborates wider event significance", "Not exact AOI asset attribution"),
], columns=["evidence_type", "source", "date", "observation", "role", "limitation"])
display(evidence_chain)

## 14. Final case-study metrics

In [ ]:
final_metrics = pd.DataFrame([
    ("AOI area", 36.00, "km²"),
    ("Semantic baseline water", reconstructed["baseline_ha"], "ha"),
    ("Near-event total water", reconstructed["event_ha"], "ha"),
    ("Mapped event water gain", reconstructed["gain_ha"], "ha"),
    ("Recovery total water", reconstructed["recovery_ha"], "ha"),
    ("Near-event to recovery mapped loss", reconstructed["loss_ha"], "ha"),
    ("Maximum local increase (G042)", grid.event_minus_baseline_pp.max(), "percentage points"),
    ("Cells above 5.88 pp context", int(grid.event_minus_baseline_pp.gt(CONTROL_CONTEXT_PP).sum()), "cells"),
    ("Directly intersecting road ways", len(direct_roads), "ways"),
    ("Direct road intersection length", direct_roads.flood_intersection_m.sum()/1000, "km"),
    ("Direct bridge-tagged intersections", int(bridges.direct_intersection.sum()), "ways"),
    ("Mapped gain intersecting ALR", alr_gain_ha, "ha"),
    ("Recovery RCM/Sentinel-2 F1", 0.790, "score"),
    ("Recovery RCM/Sentinel-2 IoU", 0.653, "score"),
    ("Recovery classification", np.nan, "recovering with localized residual/persistent expansion"),
], columns=["metric", "value", "unit_or_class"])
display(final_metrics.round(3))

## 15. Final maps for product reuse

In [ ]:
extent = [AOI_BOUNDS_UTM[0], AOI_BOUNDS_UTM[2], AOI_BOUNDS_UTM[1], AOI_BOUNDS_UTM[3]]

def finish_map(fig, ax, filename, source_note):
    ax.set_xlim(AOI_BOUNDS_UTM[0], AOI_BOUNDS_UTM[2]); ax.set_ylim(AOI_BOUNDS_UTM[1], AOI_BOUNDS_UTM[3])
    ax.set_aspect("equal"); ax.set_xlabel("Easting (m), EPSG:32610"); ax.set_ylabel("Northing (m)")
    ax.text(0.01, 0.01, source_note, transform=ax.transAxes, fontsize=7, va="bottom",
            bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"})
    fig.tight_layout(); fig.savefig(ASSET_DIR / filename, dpi=180, bbox_inches="tight"); plt.show(); plt.close(fig)

# 1 — change map
fig, ax = plt.subplots(figsize=(8.5, 7.2))
state = np.zeros(TARGET_SHAPE, dtype=np.uint8)
state[baseline] = 1; state[gain] = 2; state[loss] = 3
ax.imshow(state, extent=extent, origin="upper", cmap=ListedColormap(["white", "#3182bd", "#e34a33", "#fdbb84"]), interpolation="nearest", alpha=0.9)
ax.set_title("December 2025 Lower Fraser: mapped water change")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#3182bd", label="semantic baseline water"), Patch(color="#e34a33", label="near-event water gain"), Patch(color="#fdbb84", label="gain lost by 21 Dec")], loc="upper right")
finish_map(fig, ax, "01_flood_gain_map.png", "Source: NRCan EGS RCM-derived flood products; 20 m analytical grid")

# 2 — priority grid
fig, ax = plt.subplots(figsize=(8.5, 7.2))
gpd.GeoSeries([gain_union], crs=WORKING_CRS).plot(ax=ax, color="#fcae91", alpha=0.8)
grid.boundary.plot(ax=ax, color="#777", linewidth=0.5)
priority.plot(ax=ax, facecolor="none", edgecolor="#cb181d", linewidth=2)
for row in priority.itertuples():
    c = row.geometry.centroid; ax.text(c.x, c.y, row.grid_id, ha="center", va="center", fontsize=8, weight="bold")
ax.set_title("Priority 1 km grids: expansion and direct asset context")
finish_map(fig, ax, "02_priority_grid_map.png", "Priority = above 5.88 pp control context or direct road/bridge intersection; not a risk score")

# 3 — infrastructure
fig, ax = plt.subplots(figsize=(8.5, 7.2))
gpd.GeoSeries([gain_union], crs=WORKING_CRS).plot(ax=ax, color="#fb6a4a", alpha=0.55, label="mapped gain")
roads.plot(ax=ax, color="#969696", linewidth=0.45, label="OSM roads")
direct_roads.plot(ax=ax, color="#252525", linewidth=2, label="direct road intersections")
bridges.plot(ax=ax, color="#2ca25f", markersize=45, marker="D", label="bridge-tagged ways")
ax.legend(loc="upper right"); ax.set_title("Potential infrastructure exposure")
finish_map(fig, ax, "03_infrastructure_exposure_map.png", "Sources: NRCan EGS; © OpenStreetMap contributors, ODbL. Intersection does not prove damage.")

# 4 — community / ALR context
fig, ax = plt.subplots(figsize=(8.5, 7.2))
alr.plot(ax=ax, facecolor="#c7e9c0", edgecolor="#238b45", alpha=0.45, label="ALR designation")
gpd.GeoSeries([unary_union([shape(g) for g, v in shapes(baseline.astype(np.uint8), mask=baseline, transform=TARGET_TRANSFORM) if v == 1])], crs=WORKING_CRS).plot(ax=ax, color="#6baed6", alpha=0.75, label="baseline water context")
gpd.GeoSeries([gain_union], crs=WORKING_CRS).plot(ax=ax, color="#de2d26", alpha=0.7, label="mapped gain")
near_places = places.loc[places.distance_to_gain_m.le(5000)]
near_places.plot(ax=ax, color="#54278f", markersize=28, zorder=5, label="official place point")
for row in near_places.head(7).itertuples():
    ax.annotate(row.place_name, (row.geometry.x, row.geometry.y), xytext=(4, 4), textcoords="offset points", fontsize=7)
gpd.GeoSeries([AOI_UTM], crs=WORKING_CRS).boundary.plot(ax=ax, color="black", linewidth=1.2)
ax.legend(loc="upper right", fontsize=8); ax.set_title("Community, water-corridor and ALR context")
finish_map(fig, ax, "04_context_map.png", "Sources: ISED/NRCan place names; B.C. ALR; NRCan EGS")

## 16. Supported claims

- Official RCM-derived EGS products map water expansion inside the selected AOI.
- Approximately **54.92 ha** of additional open water was mapped near the event relative to the EGS semantic permanent-water class.
- Mapped water extent declined during the following observations; by 21 December, 35.44 ha of the near-event extent was no longer mapped as water.
- **14** clipped OSM road ways, totaling about **0.346 km** of line overlap, intersect mapped water gain.
- One OSM bridge-tagged BC 7 / Lougheed Highway way intersects mapped gain.
- **36.65 ha** of mapped gain intersects B.C. Agricultural Land Reserve designation.
- Independent Sentinel-2 imagery broadly supports the recovery-period water pattern (F1 0.790; IoU 0.653), with a 6.2-day timing offset.
- The pattern is broadly and regionally consistent with official Lower Fraser / Fraser Valley flood reporting.

## 17. Unsupported / restricted claims

- Satellite overlap does not prove a road was physically damaged, closed, or unsafe.
- The mapped water does not prove bridge damage or failure.
- The ALR overlap is not proof of cropped acreage, crop loss, or agricultural damage.
- Point proximity does not support population-exposure counts.
- The flood cannot be attributed to climate change from this case study.
- The compact AOI does not represent the full Fraser Valley event.
- The Folly Lake 5.88 percentage-point value is not universal and is not a significance test.
- OSM is not a complete infrastructure inventory.

## 18. Limitations

In [ ]:
limitations = pd.DataFrame({"limitation": [
    "EGS is operational best-effort open-water mapping and may omit urban, forested or vegetated flooding.",
    "The semantic baseline is the EGS permanent-water class, not a multi-date pre-event frequency model.",
    "Road and bridge results depend on current OSM geometry and tag completeness.",
    "ALR is a legal designation and cannot identify active crops or losses.",
    "Official place points are contextual locations, not settlement boundaries or population surfaces.",
    "Distance buffers (50/100/250 m) are descriptive and do not estimate damage probability.",
    "Official reports concern the wider region and are not exact asset-level validation inside the AOI.",
    "Sentinel-2 recovery comparison has a 6.2-day timing offset.",
]})
display(limitations)

## 19. Phase 5 findings

In [ ]:
# Contiguous mapped-gain zones, with exposure context.
gain_zones = gpd.GeoDataFrame({"zone_id": [f"Z{i:03d}" for i in range(1, len(gain_parts)+1)]}, geometry=gain_parts, crs=WORKING_CRS)
gain_zones["area_ha"] = gain_zones.area / 10_000
gain_zones["road_count_direct"] = gain_zones.geometry.apply(lambda geom: int(roads.geometry.intersects(geom).sum()))
gain_zones["bridge_count_direct"] = gain_zones.geometry.apply(lambda geom: int(bridges.geometry.intersects(geom).sum()))
gain_zones["nearest_community"] = gain_zones.geometry.apply(lambda geom: places.loc[places.geometry.distance(geom).idxmin(), "place_name"])
gain_zones["community_distance_m"] = gain_zones.geometry.apply(lambda geom: places.geometry.distance(geom).min())
gain_zones["alr_overlap_ha"] = gain_zones.geometry.apply(lambda geom: geom.intersection(alr_union).area / 10_000)
important_zones = gain_zones.loc[(gain_zones.road_count_direct > 0) | (gain_zones.bridge_count_direct > 0)].copy()
important_zones = pd.concat([important_zones, gain_zones.nlargest(10, "area_ha")]).drop_duplicates("zone_id").sort_values("area_ha", ascending=False)
display(important_zones.drop(columns="geometry").head(20).round(3))

findings = pd.DataFrame({"finding": [
    "G042 is the strongest local expansion (+21.50 pp; 5.16 ha), while G015 is second (+6.79 pp; 5.16 ha).",
    "Nine relevant grid cells are retained because they exceed the control context or contain direct road/bridge overlap.",
    "Direct road overlap is short in total (about 346 m) but spans tracks, residential, primary, unclassified and path/other classes.",
    "One bridge-tagged Lougheed Highway / BC 7 way directly intersects mapped gain; two other bridge-tagged ways do not.",
    "The nearest official place point, Leq' a: mel, is about 304 m from mapped gain; no official place point is within 250 m.",
    "About two-thirds of mapped gain intersects ALR-designated land, supporting agricultural land-use context without proving crop damage.",
    "The overall evidence supports a recovering flood with localized residual/persistent expansion.",
]})
display(findings)

## 20. Phase 6 handoff

In [ ]:
# Small, provenance-aware files for Phase 6. No raw satellite scenes are written.
final_metrics.to_csv(OUTPUT_DIR / "final_case_metrics.csv", index=False)
priority.drop(columns="geometry").to_csv(OUTPUT_DIR / "priority_grid_impact.csv", index=False)
road_export_cols = ["osm_id", "name", "ref", "highway", "road_group", "bridge", "surface", "access", "length_in_aoi_m", "flood_intersection_m", "distance_to_gain_m", "direct_intersection", "within_50m", "within_100m", "within_250m"]
roads[road_export_cols].sort_values(["direct_intersection", "distance_to_gain_m"], ascending=[False, True]).to_csv(OUTPUT_DIR / "road_exposure.csv", index=False)
bridges[road_export_cols].sort_values("distance_to_gain_m").to_csv(OUTPUT_DIR / "bridge_exposure.csv", index=False)
places[["PNuid_NLidu", "place_name", "Latitude", "Longitude", "inside_aoi", "distance_to_gain_m", "within_50m", "within_100m", "within_250m"]].head(25).to_csv(OUTPUT_DIR / "community_proximity.csv", index=False)
evidence_chain.to_csv(OUTPUT_DIR / "evidence_chain.csv", index=False)
official_impacts.to_csv(OUTPUT_DIR / "official_event_sources.csv", index=False)
limitations.to_csv(OUTPUT_DIR / "limitations.csv", index=False)
pd.DataFrame([
    ("2025-12-12", 1026.80, 54.92, "near-event"),
    ("2025-12-21", 993.52, 21.64, "recovery-period"),
], columns=["date", "total_water_ha", "flood_outside_baseline_ha", "phase"]).to_csv(OUTPUT_DIR / "temporal_series.csv", index=False)

gpd.GeoDataFrame({"feature": ["mapped_event_gain"]}, geometry=[gain_union], crs=WORKING_CRS).to_file(OUTPUT_DIR / "flood_gain.geojson", driver="GeoJSON")
priority[["grid_id", "event_minus_baseline_pp", "event_gain_area_ha", "interpretation", "geometry"]].to_file(OUTPUT_DIR / "priority_grids.geojson", driver="GeoJSON")
gain_zones.to_file(OUTPUT_DIR / "flood_gain_zones.geojson", driver="GeoJSON")
roads.loc[roads.within_250m, road_export_cols + ["geometry"]].to_file(OUTPUT_DIR / "exposed_roads.geojson", driver="GeoJSON")
if not alr_gain.is_empty:
    gpd.GeoDataFrame({"feature": ["gain_intersecting_ALR"]}, geometry=[alr_gain], crs=WORKING_CRS).to_file(OUTPUT_DIR / "alr_gain_overlap.geojson", driver="GeoJSON")

handoff = {
    "phase": 5,
    "status": "complete",
    "crs": WORKING_CRS,
    "event": "December 2025 Fraser Valley / Lower Fraser flood",
    "tables": sorted(path.name for path in OUTPUT_DIR.glob("*.csv")),
    "vectors": sorted(path.name for path in OUTPUT_DIR.glob("*.geojson")),
    "maps": sorted(path.name for path in ASSET_DIR.glob("*.png")),
    "claims_sections": ["supported claims", "unsupported / restricted claims", "limitations"],
    "phase6_note": "Use these derived outputs; do not rerun raw discovery in the dashboard.",
}
(OUTPUT_DIR / "phase6_handoff.json").write_text(json.dumps(handoff, indent=2))

for frame in [roads, bridges, places, alr, grid, gain_zones]:
    assert frame.crs.to_string() == WORKING_CRS
assert abs(direct_roads.flood_intersection_m.sum() - 345.7434) < 1
assert places.distance_to_gain_m.min() > 250
assert 36.0 < alr_gain_ha < 37.0

display(Markdown("**Phase 6 should read the derived CSV/GeoJSON files and four PNG maps listed below; it should not rerun discovery or raw satellite processing.**"))
display(pd.DataFrame({"artifact": handoff["tables"] + handoff["vectors"] + handoff["maps"]}))
print("Phase 5 handoff manifest:", OUTPUT_DIR / "phase6_handoff.json")